# 🚦 SpeedVision: Vehicle Speed Estimation with YOLO11 + ByteTrack

| Component | Technology |
|---|---|
| Detection | Ultralytics **YOLO11** (COCO pre-trained: car, motorcycle, bus, truck) |
| Tracking | **ByteTrack** multi-object tracking with persistent IDs |
| Speed engine | Homography (`cv2.getPerspectiveTransform`) → pixel-to-meter mapping → sliding-window velocity + EMA smoothing |
| Visual output | Speed-colored boxes, motion trails, bird's-eye radar minimap, live dashboard |
| Interface | **Gradio** app: click-to-calibrate, live bird's-eye check, KPI cards, interactive Plotly analytics, filterable report, CSV export |

### How to run
1. Runtime → Change runtime type → T4 GPU
2. Runtime → Run all
3. Open the public gradio.live link printed by the last code cell.

> Calibration corners are always clicked in order **top-left → top-right → bottom-right → bottom-left** of a road section whose real width and length (meters) you know.

## 1. Install dependencies

In [ ]:
!pip install -q "ultralytics>=8.3.0" "gradio>=5.9,<6" supervision lap plotly 2>&1 | grep -v -E "dependency resolver|requires .* but you have|incompatible" || true

import importlib.metadata as md
for pkg in ["ultralytics", "gradio", "supervision", "plotly"]:
    print(f"{pkg:12s} {md.version(pkg)}")

## 2. Imports & hardware check

In [ ]:
import os, json, time, shutil, subprocess, tempfile
from collections import defaultdict, deque
from dataclasses import dataclass, field, asdict

import cv2
import numpy as np
import pandas as pd
import torch
import gradio as gr
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch {torch.__version__} | Device: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else " ⚠️ enable a GPU runtime for real-time speed"))
print("ffmpeg available:", shutil.which("ffmpeg") is not None)

## 3. Configuration


In [ ]:
@dataclass
class Config:
    # --- detection / tracking ---
    model_weights: str = "yolo11m.pt"          # yolo11n/s/m/l/x — n is fastest, x most accurate
    conf_threshold: float = 0.30
    iou_threshold: float = 0.50
    imgsz: int = 1280                          # larger = better for small/far vehicles
    vehicle_classes: tuple = (2, 3, 5, 7)      # COCO: car, motorcycle, bus, truck
    tracker: str = "bytetrack.yaml"            # or "botsort.yaml"

    # --- calibration (image pixels -> real-world meters) ---
    # TL, TR, BR, BL corners of a real road rectangle in the image
    source_points: list = field(default_factory=lambda: [[1252, 787], [2298, 803], [5039, 2159], [-550, 2159]])
    target_width_m: float = 25.0               # real width of that rectangle (m)
    target_length_m: float = 250.0             # real length of that rectangle (m)

    # --- speed engine ---
    speed_unit: str = "km/h"                   # "km/h" or "mph"
    speed_limit: float = 100.0                 # vehicles above this are flagged red
    speed_window_s: float = 1.0                # displacement measured over this time window
    min_track_s: float = 0.5                   # min observation time before showing a speed
    ema_alpha: float = 0.35                    # smoothing (0 = frozen, 1 = no smoothing)

    # --- runtime ---
    max_seconds: float = 30.0                  # cap processing length (0 = full video)
    output_max_width: int = 1920               # output re-encoded to H.264 at most this wide

CFG = Config()
print(json.dumps(asdict(CFG), indent=2))

## 4. Perspective transformation (pixel → meter)


In [ ]:
class ViewTransformer:
    # Maps image pixel coordinates to real-world (meter) coordinates on the road plane.
    def __init__(self, source_points, width_m: float, length_m: float):
        src = np.asarray(source_points, dtype=np.float32)
        if src.shape != (4, 2):
            raise ValueError("Exactly 4 source points [[x,y],...] are required (TL, TR, BR, BL).")
        dst = np.array([[0, 0], [width_m, 0], [width_m, length_m], [0, length_m]], dtype=np.float32)
        self.matrix = cv2.getPerspectiveTransform(src, dst)

    def transform_points(self, points: np.ndarray) -> np.ndarray:
        if points is None or len(points) == 0:
            return np.empty((0, 2), dtype=np.float32)
        pts = np.asarray(points, dtype=np.float32).reshape(-1, 1, 2)
        return cv2.perspectiveTransform(pts, self.matrix).reshape(-1, 2)

## 5. Speed calculation engine


In [ ]:
class SpeedEstimator:
    UNIT_FACTORS = {"km/h": 3.6, "mph": 2.236936}

    def __init__(self, fps, window_s=1.0, min_track_s=0.5, ema_alpha=0.35, unit="km/h"):
        self.fps = float(fps)
        self.window = max(2, int(round(self.fps * window_s)))
        self.min_points = max(2, min(self.window, int(round(self.fps * min_track_s))))
        self.ema_alpha = ema_alpha
        self.factor = self.UNIT_FACTORS[unit]
        self.history = defaultdict(lambda: deque(maxlen=self.window))
        self.speeds = {}

    def update(self, track_id: int, frame_idx: int, point_m):
        hist = self.history[track_id]
        hist.append((frame_idx, np.asarray(point_m, dtype=np.float64)))
        if len(hist) < self.min_points:
            return None
        (f0, p0), (f1, p1) = hist[0], hist[-1]
        dt = (f1 - f0) / self.fps
        if dt <= 0:
            return self.speeds.get(track_id)
        raw = float(np.linalg.norm(p1 - p0)) / dt * self.factor
        prev = self.speeds.get(track_id)
        speed = raw if prev is None else self.ema_alpha * raw + (1 - self.ema_alpha) * prev
        self.speeds[track_id] = speed
        return speed

    def reset(self, track_id: int):
        # Called when a vehicle leaves the calibrated zone (measurements outside it are unreliable).
        self.history.pop(track_id, None)
        self.speeds.pop(track_id, None)
        return None

## 6. Video processing pipeline
Detect → track → transform → estimate → annotate → encode → report.



In [ ]:
# Colors are BGR (OpenCV). Palette follows road signage.
GREEN  = (94, 197, 34)     # #22C55E
AMBER  = (44, 199, 255)    # #FFC72C  road-sign yellow
RED    = (70, 57, 230)     # #E63946  speed-camera red
IDLE   = (156, 144, 120)   # outside zone
ASPHALT = (54, 47, 43)     # #2B2F36
WHITE  = (255, 255, 255)
FONT = cv2.FONT_HERSHEY_DUPLEX

def _scale(width):
    thick = max(2, int(round(width / 900)))
    return thick, thick * 0.32

def speed_color(speed, limit):
    if speed is None:
        return GREEN
    if not limit or limit <= 0:
        return GREEN
    r = speed / limit
    return GREEN if r < 0.75 else (AMBER if r <= 1.0 else RED)

def draw_pill(img, x, y_bottom, parts, fs, t):
    # parts: list of (text, bg_color, text_color); drawn side by side, bottom-aligned at y_bottom
    pad = max(3, int(6 * fs))
    sizes = [cv2.getTextSize(p[0], FONT, fs, t) for p in parts]
    h = max(s[0][1] + s[1] for s in sizes) + 2 * pad
    y_bottom = int(max(y_bottom, h + 2))
    x = int(max(0, x))
    for (text, bg, fg), ((tw, th), base) in zip(parts, sizes):
        cv2.rectangle(img, (x, y_bottom - h), (x + tw + 2 * pad, y_bottom), bg, -1)
        cv2.putText(img, text, (x + pad, y_bottom - pad - base // 2), FONT, fs, fg, t, cv2.LINE_AA)
        x += tw + 2 * pad
    return h

def draw_panel(img, x, y, w, h, alpha=0.6, color=ASPHALT):
    x2, y2 = min(img.shape[1], x + w), min(img.shape[0], y + h)
    roi = img[y:y2, x:x2]
    img[y:y2, x:x2] = cv2.addWeighted(roi, 1 - alpha, np.full_like(roi, color), alpha, 0)

def draw_dashed_vline(img, x, y1, y2, color, t, dash=10, gap=8):
    y = y1
    while y < y2:
        cv2.line(img, (x, y), (x, min(y + dash, y2)), color, t)
        y += dash + gap

def draw_hud(img, lines, fs, t):
    pad = int(14 * fs) + 6
    sizes = [cv2.getTextSize(txt, FONT, fs * 1.1, t)[0] for txt, _ in lines]
    lh = max(s[1] for s in sizes) + pad
    w = max(s[0] for s in sizes) + 2 * pad + int(30 * fs)
    h = lh * len(lines) + pad
    draw_panel(img, 12, 12, w, h, 0.65)
    for i, ((txt, col), _) in enumerate(zip(lines, sizes)):
        y = 12 + pad + lh * i + sizes[i][1]
        cv2.circle(img, (12 + pad + int(8 * fs), y - sizes[i][1] // 2), max(3, int(8 * fs)), col, -1)
        cv2.putText(img, txt, (12 + pad + int(30 * fs), y), FONT, fs * 1.1, WHITE, t, cv2.LINE_AA)

def draw_radar(img, dots, width_m, length_m, fs, t):
    H, W = img.shape[:2]
    rh = int(H * 0.42)
    rw = int(max(rh * width_m / length_m, W * 0.08))
    m = int(W * 0.015)
    x0, y0 = W - rw - m, m + int(40 * fs)
    draw_panel(img, x0 - 6, y0 - int(40 * fs), rw + 12, rh + int(40 * fs) + 6, 0.7)
    cv2.putText(img, "Bird's-eye", (x0, y0 - int(12 * fs)), FONT, fs * 0.9, AMBER, max(1, t - 1), cv2.LINE_AA)
    cv2.rectangle(img, (x0, y0), (x0 + rw, y0 + rh), (90, 90, 90), max(1, t // 2))
    for k in (1, 2):
        draw_dashed_vline(img, x0 + rw * k // 3, y0, y0 + rh, (200, 200, 200), max(1, t // 2),
                          dash=max(6, rh // 20), gap=max(4, rh // 30))
    for (xm, ym), col in dots:
        px = x0 + int(np.clip(xm / width_m, 0, 1) * rw)
        py = y0 + int(np.clip(ym / length_m, 0, 1) * rh)
        cv2.circle(img, (px, py), max(4, t * 2), col, -1)
        cv2.circle(img, (px, py), max(4, t * 2), WHITE, 1)

def encode_h264(src, dst, max_width):
    if shutil.which("ffmpeg") is None:
        return src
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src,
           "-vf", f"scale='min({max_width},iw)':-2",
           "-c:v", "libx264", "-preset", "veryfast", "-crf", "23",
           "-pix_fmt", "yuv420p", "-movflags", "+faststart", dst]
    try:
        subprocess.run(cmd, check=True)
        return dst
    except subprocess.CalledProcessError:
        return src

def process_video(video_path, source_points, width_m, length_m,
                  model_weights=None, conf=None, unit=None, speed_limit=None,
                  max_seconds=None, show_trails=True, show_radar=True, progress_cb=None):
    model_weights = model_weights or CFG.model_weights
    conf = CFG.conf_threshold if conf is None else conf
    unit = unit or CFG.speed_unit
    speed_limit = CFG.speed_limit if speed_limit is None else speed_limit
    max_seconds = CFG.max_seconds if max_seconds is None else max_seconds

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    if not (1 < fps <= 240):
        fps = 30.0
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    n_frames = total if total > 0 else 10**9
    if max_seconds and max_seconds > 0:
        n_frames = min(n_frames, int(fps * max_seconds))
    denom = n_frames if n_frames < 10**9 else None

    model = YOLO(model_weights)          # fresh instance => fresh tracker state per video
    vt = ViewTransformer(source_points, width_m, length_m)
    est = SpeedEstimator(fps, CFG.speed_window_s, CFG.min_track_s, CFG.ema_alpha, unit)
    polygon = np.asarray(source_points, dtype=np.int32)
    polygon_f = polygon.astype(np.float32)

    out_dir = tempfile.mkdtemp(prefix="speedvision_")
    raw_path = os.path.join(out_dir, "raw.mp4")
    writer = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    thick, fs = _scale(w)
    lt = max(1, thick // 2)

    stats, seen, violators = {}, set(), set()
    trails = defaultdict(lambda: deque(maxlen=int(fps)))
    timeline = []
    idx, t0 = 0, time.time()
    while idx < n_frames:
        ok, frame = cap.read()
        if not ok:
            break
        result = model.track(frame, persist=True, tracker=CFG.tracker, conf=conf,
                             iou=CFG.iou_threshold, classes=list(CFG.vehicle_classes),
                             imgsz=CFG.imgsz, device=DEVICE, verbose=False)[0]

        overlay = frame.copy()
        cv2.fillPoly(overlay, [polygon], AMBER)
        frame = cv2.addWeighted(overlay, 0.10, frame, 0.90, 0)
        cv2.polylines(frame, [polygon], True, AMBER, thick, cv2.LINE_AA)

        radar_dots, active = [], set()
        boxes = result.boxes
        if boxes is not None and boxes.id is not None and len(boxes) > 0:
            xyxy = boxes.xyxy.cpu().numpy()
            ids = boxes.id.int().cpu().numpy()
            clss = boxes.cls.int().cpu().numpy()
            anchors = np.stack([(xyxy[:, 0] + xyxy[:, 2]) / 2, xyxy[:, 3]], axis=1)  # bottom-center
            anchors_m = vt.transform_points(anchors)

            for box, tid, cls_id, a_px, a_m in zip(xyxy, ids, clss, anchors, anchors_m):
                tid, name = int(tid), result.names[int(cls_id)]
                seen.add(tid)
                active.add(tid)
                inside = cv2.pointPolygonTest(polygon_f, (float(a_px[0]), float(a_px[1])), False) >= 0
                speed = est.update(tid, idx, a_m) if inside else est.reset(tid)

                color = speed_color(speed, speed_limit) if inside else IDLE
                if speed is not None:
                    s = stats.setdefault(tid, {"class": name, "speeds": [], "first": idx, "last": idx})
                    s["speeds"].append(speed)
                    s["last"] = idx
                    if idx % 3 == 0:
                        timeline.append((round(idx / fps, 2), tid, name, round(speed, 1)))
                    if speed_limit and speed > speed_limit:
                        violators.add(tid)
                if inside:
                    radar_dots.append(((float(a_m[0]), float(a_m[1])), color))

                trails[tid].append((int(a_px[0]), int(a_px[1])))
                if show_trails and len(trails[tid]) > 1:
                    pts = list(trails[tid])
                    for i in range(1, len(pts)):
                        cv2.line(frame, pts[i - 1], pts[i], color,
                                 max(1, int(thick * 1.5 * i / len(pts))), cv2.LINE_AA)

                x1, y1, x2, y2 = box.astype(int)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, thick, cv2.LINE_AA)
                cv2.circle(frame, (int(a_px[0]), int(a_px[1])), thick + 2, color, -1)
                parts = [(f"#{tid} {name}", ASPHALT, WHITE)]
                if speed is not None:
                    txt_col = ASPHALT if color in (GREEN, AMBER) else WHITE
                    parts.append((f"{speed:.0f} {unit}", color, txt_col))
                draw_pill(frame, x1, y1 - 2, parts, fs, lt)

        for tid in list(trails):               # drop trails of vehicles no longer visible
            if tid not in active:
                del trails[tid]

        if show_radar:
            draw_radar(frame, radar_dots, width_m, length_m, fs, thick)

        elapsed = max(time.time() - t0, 1e-6)
        draw_hud(frame, [
            (f"Vehicles  {len(seen)}", WHITE),
            (f"Measured  {len(stats)}", GREEN),
            (f"Over {speed_limit:.0f} {unit}  {len(violators)}", RED),
            (f"{(idx + 1) / elapsed:.1f} FPS", AMBER),
        ], fs, lt)

        writer.write(frame)
        idx += 1
        if progress_cb and idx % 5 == 0:
            progress_cb(min(idx / denom, 1.0) if denom else 0.5)

    cap.release()
    writer.release()
    if idx == 0:
        raise ValueError("No frames could be read from the video.")

    out_path = encode_h264(raw_path, os.path.join(out_dir, "speedvision_output.mp4"), CFG.output_max_width)

    rows = []
    for tid, s in sorted(stats.items()):
        sp = np.asarray(s["speeds"])
        rows.append({
            "Vehicle ID": tid,
            "Class": s["class"],
            f"Avg speed ({unit})": round(float(sp.mean()), 1),
            f"Max speed ({unit})": round(float(sp.max()), 1),
            "Time in zone (s)": round((s["last"] - s["first"] + 1) / fps, 2),
            "Over limit": "Yes" if tid in violators else "No",
        })
    report = pd.DataFrame(rows, columns=["Vehicle ID", "Class", f"Avg speed ({unit})", f"Max speed ({unit})",
                                         "Time in zone (s)", "Over limit"])
    csv_path = os.path.join(out_dir, "speed_report.csv")
    report.to_csv(csv_path, index=False)
    timeline_df = pd.DataFrame(timeline, columns=["Time (s)", "Vehicle ID", "Class", "Speed"])

    elapsed = time.time() - t0
    summary = {
        "frames_processed": idx,
        "video_seconds": round(idx / fps, 1),
        "processing_fps": round(idx / max(elapsed, 1e-6), 1),
        "vehicles_tracked": len(seen),
        "vehicles_measured": len(stats),
        "speed_violations": len(violators),
        "mean_speed": round(float(report[f"Avg speed ({unit})"].mean()), 1) if len(report) else None,
        "max_speed": round(float(report[f"Max speed ({unit})"].max()), 1) if len(report) else None,
        "unit": unit,
        "speed_limit": speed_limit,
        "resolution": f"{w}x{h}",
        "fps": round(fps, 2),
    }
    return out_path, report, csv_path, summary, timeline_df

## 7. Analytics charts


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

C_GREEN, C_AMBER, C_RED, C_SIGN, C_BLUE = "#22C55E", "#FFC72C", "#E63946", "#0B6E4F", "#3B82F6"
CLASS_COLORS = {"car": C_BLUE, "truck": C_AMBER, "bus": C_SIGN, "motorcycle": C_RED}

def _band(v, limit):
    if not limit:
        return C_GREEN
    r = v / limit
    return C_GREEN if r < 0.75 else (C_AMBER if r <= 1.0 else C_RED)

LINE_COLORS = [C_BLUE, C_AMBER, C_SIGN, C_RED, "#8B5CF6", "#06B6D4", "#F97316", "#22C55E", "#EC4899", "#64748B"]

def _style(fig, title):
    fig.update_layout(
        title=dict(text=title, font=dict(size=16)), height=340,
        margin=dict(l=10, r=10, t=50, b=10),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        font=dict(family="Overpass, sans-serif", color="#8A94A3"),
        legend=dict(orientation="h", y=-0.2),
    )
    fig.update_xaxes(gridcolor="rgba(128,128,128,0.18)", zeroline=False)
    fig.update_yaxes(gridcolor="rgba(128,128,128,0.18)", zeroline=False)
    return fig

def _empty(title):
    fig = go.Figure()
    fig.add_annotation(text="Run an analysis to fill this chart", showarrow=False, font=dict(size=14))
    fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
    return _style(fig, title)

def make_charts(report, timeline, unit, limit):
    avg_col, max_col = f"Avg speed ({unit})", f"Max speed ({unit})"
    if report is None or len(report) == 0:
        return (_empty("Speed distribution"), _empty("Top speed per vehicle"),
                _empty("Vehicle mix"), _empty("Speed over time"))

    hist = px.histogram(report, x=avg_col, color="Over limit", nbins=20,
                        color_discrete_map={"No": C_GREEN, "Yes": C_RED})
    if limit:
        hist.add_vline(x=limit, line_dash="dash", line_color=C_RED,
                       annotation_text=f"Limit {limit:.0f}", annotation_position="top")
    hist.update_layout(bargap=0.08, yaxis_title="Vehicles")

    top = report.sort_values(max_col, ascending=False).head(25)
    bar = go.Figure(go.Bar(
        x=[f"#{i}" for i in top["Vehicle ID"]], y=top[max_col],
        marker=dict(color=[_band(v, limit) for v in top[max_col]]),
        hovertext=top["Class"], hovertemplate="%{x} %{hovertext}<br>%{y:.1f} " + unit + "<extra></extra>"))
    if limit:
        bar.add_hline(y=limit, line_dash="dash", line_color=C_RED)
    bar.update_layout(yaxis_title=unit)

    counts = report["Class"].value_counts().reset_index()
    counts.columns = ["Class", "Count"]
    donut = px.pie(counts, names="Class", values="Count", hole=0.6,
                   color="Class", color_discrete_map=CLASS_COLORS)
    donut.update_traces(textinfo="label+value")

    fastest = report.sort_values(max_col, ascending=False)["Vehicle ID"].head(10).tolist()
    tl = timeline[timeline["Vehicle ID"].isin(fastest)].copy()
    tl["Vehicle"] = "#" + tl["Vehicle ID"].astype(str) + " " + tl["Class"]
    line = px.line(tl, x="Time (s)", y="Speed", color="Vehicle", color_discrete_sequence=LINE_COLORS)
    if limit:
        line.add_hline(y=limit, line_dash="dash", line_color=C_RED)
    line.update_layout(yaxis_title=unit)

    return (_style(hist, "Speed distribution"), _style(bar, "Top speed per vehicle"),
            _style(donut, "Vehicle mix"), _style(line, "Speed over time (10 fastest)"))

## 8. Download sample video & smoke test (optional but recommended)


In [ ]:
from supervision.assets import download_assets, VideoAssets

SAMPLE_VIDEO = None
try:
    SAMPLE_VIDEO = os.path.abspath(download_assets(VideoAssets.VEHICLES))
    print("Sample video:", SAMPLE_VIDEO)
except Exception as e:
    print("Sample download failed (you can still upload your own video):", e)

if SAMPLE_VIDEO:
    out_path, report, csv_path, summary, timeline = process_video(
        SAMPLE_VIDEO, CFG.source_points, CFG.target_width_m, CFG.target_length_m,
        model_weights="yolo11m.pt", max_seconds=10)
    print(json.dumps(summary, indent=2))
    display(report.head(15))

In [ ]:
from IPython.display import Video
if SAMPLE_VIDEO:
    display(Video(out_path, embed=True, width=900))

## 9. SpeedVision web app


In [ ]:
SIGN_GREEN, SIGN_YELLOW, CAM_RED, ASPHALT_HEX = "#0B6E4F", "#FFC72C", "#E63946", "#2B2F36"
CORNER_NAMES = ["top-left", "top-right", "bottom-right", "bottom-left"]

CSS = '''
.gradio-container { max-width: 1400px !important; }
.sv-hero { display:flex; align-items:center; gap:28px; background:#0B6E4F; color:#fff;
  border:4px solid #fff; outline:3px solid #0B6E4F; border-radius:16px; padding:22px 30px; margin:6px 3px 0; }
.sv-hero h1 { margin:0; font-weight:800; font-size:2.7rem; line-height:1.05; color:#fff; letter-spacing:-0.01em; }
.sv-hero p { margin:8px 0 0; font-size:1.05rem; color:rgba(255,255,255,.88); max-width:64ch; line-height:1.45; }
.sv-sign { flex:none; width:104px; height:104px; border-radius:50%; background:#fff; border:13px solid #E63946;
  display:grid; place-items:center; color:#1d2026; font-weight:800; font-size:2.2rem; }
.sv-lane { height:7px; margin:16px 3px 4px; border-radius:4px;
  background:repeating-linear-gradient(90deg,#FFC72C 0 46px,transparent 46px 78px); }
.sv-steps { display:flex; flex-wrap:wrap; gap:8px; align-items:center; margin:2px 0 4px; }
.sv-chip { padding:4px 12px; border-radius:999px; border:2px solid #FFC72C; font-weight:700; font-size:.9rem; }
.sv-chip.done { background:#FFC72C; color:#2B2F36; }
.sv-chip.next { box-shadow:0 0 0 3px rgba(255,199,44,.35); }
.sv-hint { margin:6px 0 0; font-size:.98rem; }
.sv-kpis { display:grid; grid-template-columns:1.15fr 1fr 1fr; gap:12px; }
.sv-kpi { border-radius:12px; padding:14px 16px; border:1px solid var(--border-color-primary);
  background:var(--block-background-fill); border-left:7px solid var(--accent); }
.sv-kpi .v { font-size:2.1rem; font-weight:800; line-height:1.1; }
.sv-kpi .l { font-size:.9rem; opacity:.75; margin-top:2px; }
.sv-kpi.top { grid-row:span 2; display:flex; flex-direction:column; align-items:center; justify-content:center;
  text-align:center; gap:10px; border-left:1px solid var(--border-color-primary); border-top:7px solid #E63946; }
.sv-kpi.top .sv-sign { width:104px; height:104px; border-width:12px; font-size:2rem; }
.sv-kpi.top .t { font-weight:800; font-size:1.15rem; }
.sv-foot { grid-column:1 / -1; font-size:.88rem; opacity:.72; margin:2px 2px 0; }
@media (max-width: 720px) { .sv-kpis { grid-template-columns:1fr 1fr; } .sv-kpi.top { grid-row:auto; grid-column:1 / -1; } }
.sv-empty { padding:22px; border:2px dashed var(--border-color-primary); border-radius:12px; text-align:center; opacity:.8; }
#sv-run { font-size:1.15rem; font-weight:800; min-height:54px; }
@media (max-width: 720px) { .sv-hero { flex-direction:column; align-items:flex-start; } .sv-hero h1 { font-size:2rem; } }
'''

THEME = gr.themes.Base(
    primary_hue=gr.themes.colors.amber, secondary_hue=gr.themes.colors.emerald,
    neutral_hue=gr.themes.colors.slate, radius_size=gr.themes.sizes.radius_md,
    font=[gr.themes.GoogleFont("Overpass"), "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    button_primary_background_fill=SIGN_YELLOW, button_primary_background_fill_hover="#F5B700",
    button_primary_text_color=ASPHALT_HEX,
    button_primary_background_fill_dark=SIGN_YELLOW, button_primary_background_fill_hover_dark="#F5B700",
    button_primary_text_color_dark=ASPHALT_HEX,
    slider_color=SIGN_YELLOW, slider_color_dark=SIGN_YELLOW,
    block_title_text_weight="700",
)

HERO = f'''
<div class="sv-hero">
  <div class="sv-sign">{int(CFG.speed_limit)}</div>
  <div>
    <h1>SpeedVision</h1>
    <p>Measure the speed of every vehicle in a traffic video. YOLO11 finds the vehicles, ByteTrack follows them
    frame to frame, and your road calibration turns pixels into real meters.</p>
  </div>
</div>
<div class="sv-lane"></div>
'''

# ---------- helpers ----------
def parse_points(text, strict=True):
    try:
        pts = json.loads(text) if isinstance(text, str) else text
        pts = [[float(p[0]), float(p[1])] for p in pts]
        if len(pts) != 4:
            raise ValueError
        return pts
    except Exception:
        if strict:
            raise gr.Error("Place 4 calibration points first: click the frame, or type JSON like [[x1,y1],[x2,y2],[x3,y3],[x4,y4]].")
        return []

def status_html(n):
    chips = "".join(
        f'<span class="sv-chip {"done" if i < n else ("next" if i == n else "")}">{i + 1}. {name.capitalize()}</span>'
        for i, name in enumerate(CORNER_NAMES))
    if n == 0:
        hint = "Click the <b>top-left</b> corner of a road section whose real size you know (lane markings work well)."
    elif n < 4:
        hint = f"Now click the <b>{CORNER_NAMES[n]}</b> corner."
    else:
        hint = "Calibration set. Check the bird's-eye view, then run the analysis."
    return f'<div class="sv-steps">{chips}</div><p class="sv-hint">{hint}</p>'

def draw_points(img, pts):
    canvas = img.copy()
    thick, fs = _scale(canvas.shape[1])
    yellow, red = (255, 199, 44), (230, 57, 70)   # RGB here (Gradio images are RGB)
    if len(pts) == 4:
        overlay = canvas.copy()
        cv2.fillPoly(overlay, [np.asarray(pts, dtype=np.int32)], yellow)
        canvas = cv2.addWeighted(overlay, 0.18, canvas, 0.82, 0)
    if len(pts) >= 2:
        cv2.polylines(canvas, [np.asarray(pts, dtype=np.int32)], len(pts) == 4, yellow, thick + 1, cv2.LINE_AA)
    for i, (x, y) in enumerate(pts):
        r = thick * 7
        cv2.circle(canvas, (int(x), int(y)), r, red, -1, cv2.LINE_AA)
        cv2.circle(canvas, (int(x), int(y)), r, (255, 255, 255), thick, cv2.LINE_AA)
        (tw, th), _ = cv2.getTextSize(str(i + 1), FONT, fs * 2, thick)
        cv2.putText(canvas, str(i + 1), (int(x) - tw // 2, int(y) + th // 2), FONT, fs * 2,
                    (255, 255, 255), thick, cv2.LINE_AA)
    return canvas

def birdseye(base, pts, width_m, length_m):
    if base is None or len(pts) != 4 or not width_m or not length_m or width_m <= 0 or length_m <= 0:
        return None
    ppm = max(2.0, min(20.0, 900.0 / max(width_m, length_m)))
    W, L = max(1, int(width_m * ppm)), max(1, int(length_m * ppm))
    dst = np.array([[0, 0], [W, 0], [W, L], [0, L]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(np.asarray(pts, dtype=np.float32), dst)
    warped = cv2.warpPerspective(base, M, (W, L))
    pad = 46
    canvas = np.full((L, W + pad, 3), (43, 47, 54), dtype=np.uint8)
    canvas[:, pad:] = warped
    step = 10 if length_m <= 300 else 50
    for mtr in range(0, int(length_m) + 1, step):
        y = min(L - 1, int(mtr * ppm))
        cv2.line(canvas, (pad, y), (pad + W, y), (255, 199, 44), 1)
        cv2.putText(canvas, f"{mtr}m", (3, max(12, y + 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 199, 44), 1, cv2.LINE_AA)
    return canvas

def kpi_html(s):
    if not s:
        return '<div class="sv-empty">Results will appear here after you run an analysis.</div>'
    u = s["unit"]
    top = "–" if s["max_speed"] is None else f'{s["max_speed"]:.0f}'
    mean = "–" if s["mean_speed"] is None else f'{s["mean_speed"]:.0f}'
    def card(v, l, accent):
        return f'<div class="sv-kpi" style="--accent:{accent}"><div class="v">{v}</div><div class="l">{l}</div></div>'
    return (
        '<div class="sv-kpis">'
        f'<div class="sv-kpi top"><div class="sv-sign">{top}</div>'
        f'<div><div class="t">Top speed</div><div class="l">{u}, fastest vehicle</div></div></div>'
        + card(s["vehicles_tracked"], "Vehicles tracked", "#3B82F6")
        + card(s["vehicles_measured"], "Speeds measured", SIGN_GREEN)
        + card(mean, f"Mean speed ({u})", SIGN_YELLOW)
        + card(s["speed_violations"], f'Over {s["speed_limit"]:.0f} {u}', CAM_RED)
        + f'<div class="sv-foot">Analysed {s["frames_processed"]} frames ({s["video_seconds"]} s of {s["resolution"]} video at '
          f'{s["fps"]} fps) at {s["processing_fps"]} FPS on {DEVICE.upper()}.</div>'
        + '</div>')

def _style_report(df):
    if df is None or len(df) == 0:
        return df
    return df.style.apply(
        lambda r: ["background-color: rgba(230,57,70,.18)" if r["Over limit"] == "Yes" else "" for _ in r], axis=1)

# ---------- event handlers ----------
def on_video(video_path, points_text, width_m, length_m):
    if not video_path:
        return None, None, [], "", status_html(0), None
    cap = cv2.VideoCapture(video_path)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise gr.Error("This video could not be read. Try an MP4 (H.264) file.")
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    is_sample = "vehicles" in os.path.basename(video_path).lower()
    pts = parse_points(points_text, strict=False) if is_sample else []
    text = json.dumps([[int(x), int(y)] for x, y in pts]) if pts else ""
    return draw_points(rgb, pts), rgb, pts, text, status_html(len(pts)), birdseye(rgb, pts, width_m, length_m)

def on_click(base, pts, width_m, length_m, evt: gr.SelectData):
    if base is None:
        raise gr.Error("Upload a video first.")
    pts = [] if len(pts) >= 4 else list(pts)
    pts.append([int(evt.index[0]), int(evt.index[1])])
    bev = birdseye(base, pts, width_m, length_m) if len(pts) == 4 else None
    return draw_points(base, pts), pts, json.dumps(pts), status_html(len(pts)), bev

def on_undo(base, pts, width_m, length_m):
    pts = list(pts)[:-1]
    img = draw_points(base, pts) if base is not None else None
    return img, pts, json.dumps(pts) if pts else "", status_html(len(pts)), None

def on_reset(base):
    return base, [], "", status_html(0), None

def on_points_typed(base, text, width_m, length_m):
    pts = parse_points(text, strict=False)
    if base is None:
        return None, pts, status_html(len(pts)), None
    return draw_points(base, pts), pts, status_html(len(pts)), birdseye(base, pts, width_m, length_m)

def on_dims(base, pts, width_m, length_m):
    return birdseye(base, pts, width_m, length_m)

def on_sample():
    if not SAMPLE_VIDEO:
        raise gr.Error("The sample video is not available. Run the download cell, or upload your own video.")
    return SAMPLE_VIDEO, json.dumps(CFG.source_points), CFG.target_width_m, CFG.target_length_m

def on_run(video_path, points_text, width_m, length_m, model_name, conf, unit, limit, max_s,
           trails, radar, progress=gr.Progress()):
    if not video_path:
        raise gr.Error("Upload a video first.")
    pts = parse_points(points_text)
    if not width_m or not length_m or width_m <= 0 or length_m <= 0:
        raise gr.Error("Enter the real width and length of the calibrated road section in meters.")
    progress(0, desc="Loading model")
    out, report, csv_path, s, timeline = process_video(
        video_path, pts, float(width_m), float(length_m), model_weights=model_name,
        conf=float(conf), unit=unit, speed_limit=float(limit), max_seconds=float(max_s),
        show_trails=trails, show_radar=radar,
        progress_cb=lambda f: progress(f, desc="Detecting, tracking and measuring"))
    charts = make_charts(report, timeline, unit, float(limit))
    classes = ["All classes"] + sorted(report["Class"].unique().tolist()) if len(report) else ["All classes"]
    gr.Info(f"Analysis complete: {s['vehicles_measured']} vehicles measured.")
    return (out, kpi_html(s), _style_report(report), csv_path, *charts, report,
            gr.update(choices=classes, value="All classes"), False)

def on_filter(report, cls, only_over):
    if report is None or len(report) == 0:
        return report
    df = report
    if cls and cls != "All classes":
        df = df[df["Class"] == cls]
    if only_over:
        df = df[df["Over limit"] == "Yes"]
    return _style_report(df)

# ---------- layout ----------
with gr.Blocks(title="SpeedVision — Vehicle Speed Estimation", theme=THEME, css=CSS) as demo:
    gr.HTML(HERO)
    base_frame = gr.State(None)
    points_state = gr.State([])
    report_state = gr.State(None)

    with gr.Row(equal_height=False):
        # ----- left: input & calibration -----
        with gr.Column(scale=5):
            with gr.Group():
                video_in = gr.Video(label="Traffic video", sources=["upload"], height=260)
                sample_btn = gr.Button("Load the sample highway clip", variant="secondary", size="sm")
            status = gr.HTML(status_html(0))
            frame_view = gr.Image(label="Click the road corners on this frame", type="numpy",
                                  interactive=False, height=360, show_download_button=False)
            with gr.Row():
                undo_btn = gr.Button("Undo last point", size="sm")
                reset_btn = gr.Button("Clear points", size="sm")
            with gr.Row():
                width_in = gr.Number(label="Real width (m)", value=CFG.target_width_m, minimum=0.1)
                length_in = gr.Number(label="Real length (m)", value=CFG.target_length_m, minimum=0.1)
            points_text = gr.Textbox(label="Corner coordinates in pixels (editable, press Enter to apply)",
                                     placeholder="[[x1,y1],[x2,y2],[x3,y3],[x4,y4]]")
            with gr.Accordion("Detection and display settings", open=False):
                model_in = gr.Dropdown(["yolo11n.pt", "yolo11s.pt", "yolo11m.pt", "yolo11l.pt", "yolo11x.pt"],
                                       value=CFG.model_weights, label="Detector (n = fastest, x = most accurate)")
                conf_in = gr.Slider(0.1, 0.9, value=CFG.conf_threshold, step=0.05, label="Detection confidence")
                with gr.Row():
                    unit_in = gr.Radio(["km/h", "mph"], value=CFG.speed_unit, label="Speed unit")
                    limit_in = gr.Slider(10, 200, value=CFG.speed_limit, step=5, label="Speed limit")
                maxs_in = gr.Slider(0, 300, value=CFG.max_seconds, step=5,
                                    label="Seconds of video to analyse (0 = whole video)")
                with gr.Row():
                    trails_in = gr.Checkbox(value=True, label="Motion trails")
                    radar_in = gr.Checkbox(value=True, label="Bird's-eye radar")
            run_btn = gr.Button("Run speed analysis", variant="primary", elem_id="sv-run")

        # ----- right: results -----
        with gr.Column(scale=7):
            kpis = gr.HTML(kpi_html(None))
            with gr.Tabs():
                with gr.Tab("Annotated video"):
                    video_out = gr.Video(label="Output", height=460, autoplay=True)
                with gr.Tab("Analytics"):
                    with gr.Row():
                        plot_hist = gr.Plot(make_charts(None, None, "km/h", 0)[0], show_label=False)
                        plot_bar = gr.Plot(make_charts(None, None, "km/h", 0)[1], show_label=False)
                    with gr.Row():
                        plot_donut = gr.Plot(make_charts(None, None, "km/h", 0)[2], show_label=False)
                        plot_line = gr.Plot(make_charts(None, None, "km/h", 0)[3], show_label=False)
                with gr.Tab("Vehicle report"):
                    with gr.Row():
                        class_filter = gr.Dropdown(["All classes"], value="All classes", label="Class", scale=2)
                        over_filter = gr.Checkbox(value=False, label="Only vehicles over the limit", scale=1)
                    report_out = gr.Dataframe(wrap=True, interactive=False)
                    csv_out = gr.File(label="Download CSV report")
                with gr.Tab("Calibration check"):
                    gr.Markdown("A top-down view of your calibrated road section with 10 m markers. "
                                "Lane lines should run straight and parallel. If they fan out or bend, "
                                "re-click the corners or adjust the real width and length.")
                    bev_out = gr.Image(label="Bird's-eye view", height=520, show_download_button=False)
                with gr.Tab("How it works"):
                    gr.Markdown('''
**Detection.** Each frame goes through a COCO-pretrained YOLO11 model, keeping only cars, motorcycles, buses and trucks.

**Tracking.** ByteTrack links detections across frames so each vehicle keeps the same ID.

**Calibration.** The four corners you click define a homography that maps the camera view onto a flat,
top-down rectangle measured in meters. The point where a vehicle meets the road (bottom-center of its box) is projected through it.

**Speed.** For each vehicle, distance travelled over a 1-second sliding window is divided by elapsed time
(from frame numbers and the video's FPS), then smoothed with an exponential moving average. Speeds are measured only inside the calibrated zone.
''')

    # ----- events -----
    video_in.change(on_video, [video_in, points_text, width_in, length_in],
                    [frame_view, base_frame, points_state, points_text, status, bev_out])
    sample_btn.click(on_sample, None, [video_in, points_text, width_in, length_in])
    frame_view.select(on_click, [base_frame, points_state, width_in, length_in],
                      [frame_view, points_state, points_text, status, bev_out])
    undo_btn.click(on_undo, [base_frame, points_state, width_in, length_in],
                   [frame_view, points_state, points_text, status, bev_out])
    reset_btn.click(on_reset, [base_frame], [frame_view, points_state, points_text, status, bev_out])
    points_text.submit(on_points_typed, [base_frame, points_text, width_in, length_in],
                       [frame_view, points_state, status, bev_out])
    width_in.change(on_dims, [base_frame, points_state, width_in, length_in], [bev_out])
    length_in.change(on_dims, [base_frame, points_state, width_in, length_in], [bev_out])
    run_btn.click(on_run,
                  [video_in, points_text, width_in, length_in, model_in, conf_in, unit_in, limit_in, maxs_in,
                   trails_in, radar_in],
                  [video_out, kpis, report_out, csv_out, plot_hist, plot_bar, plot_donut, plot_line,
                   report_state, class_filter, over_filter])
    class_filter.change(on_filter, [report_state, class_filter, over_filter], [report_out])
    over_filter.change(on_filter, [report_state, class_filter, over_filter], [report_out])

if os.environ.get("SV_NO_LAUNCH") != "1":
    demo.queue().launch(share=True, debug=True)

## 10. Calibration & accuracy notes
- **Pick a real rectangle on the road plane.** Lane markings are ideal: standard lane width is about 3.5–3.7 m, and dashed markings follow fixed lengths set by local road standards.
- Points may lie **outside the frame** (negative or beyond the width) as long as they describe the road plane. The sample calibration does this; edit the coordinates box directly for such points.
- Speeds are measured **only inside the calibrated zone** (yellow area). Vehicles outside are tracked but not timed.
- Accuracy depends mostly on calibration quality. Use the *Calibration check* tab: lane lines should be straight and parallel.
- For a live camera, pass an RTSP URL (`rtsp://...`) as `video_path` to `process_video` and set `max_seconds`.